In [2]:
import nbformat as nbf
from pathlib import Path
import os

nb = nbf.v4.new_notebook()
cells = []

cells.append(nbf.v4.new_markdown_cell(
"""# 03 Time Series Feature Diagnostics

This notebook inspects the **time-series behaviour** of the macroeconomic indicators used in the downturn-risk project.

## Purpose

The goal is to make the modelling inputs more interpretable by visualising:

- **raw indicator levels**
- **year-to-year changes (delta features)**
- **3-year least-squares slopes (trend features)**
- **downturn years**, defined here as years where **next-year GDP growth is negative**

This notebook helps answer:

1. What macroeconomic state is a country currently in?
2. Are conditions improving or deteriorating?
3. Do downturn periods appear after sustained negative trends rather than only low levels?

## Why this matters

The predictive model uses:

- current levels
- lag features
- delta features
- 3-year trend features

These plots help justify that feature engineering pipeline in a transparent way.
"""
))

cells.append(nbf.v4.new_markdown_cell(
"""## Expected file location

This notebook expects one of these files to exist:

- `data/worldbank_panel_final.csv`
- `data/worldbank_panel.csv`

It is designed to run from the **project root** or from inside a `notebooks/` folder.
"""
))

cells.append(nbf.v4.new_code_cell(
"""from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

current = Path.cwd().resolve()
project_root = current.parent if current.name == "notebooks" else current

panel_final_path = project_root / "data" / "worldbank_panel_final.csv"
panel_fallback_path = project_root / "data" / "worldbank_panel.csv"

if panel_final_path.exists():
    panel_path = panel_final_path
elif panel_fallback_path.exists():
    panel_path = panel_fallback_path
else:
    raise FileNotFoundError(
        "Could not find input panel CSV. Expected one of:\\n"
        f"- {panel_final_path}\\n"
        f"- {panel_fallback_path}"
    )

print("Using panel file:", panel_path)
"""
))

cells.append(nbf.v4.new_code_cell(
"""panel_df = pd.read_csv(panel_path)
panel_df = panel_df.sort_values(["country_code", "year"]).reset_index(drop=True)

required_cols = [
    "country_code",
    "year",
    "unemployment",
    "inflation",
    "gdp_growth",
    "life_expectancy",
    "population_growth",
]

missing_cols = [c for c in required_cols if c not in panel_df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

if "country" not in panel_df.columns:
    panel_df["country"] = panel_df["country_code"]

panel_df.head()
"""
))

cells.append(nbf.v4.new_markdown_cell(
"""## Construct delta and trend features

We now derive:

- annual change features:  
  \\[
  \\Delta x_t = x_t - x_{t-1}
  \\]

- 3-year trend features using least-squares slope:  
  \\[
  \\hat{\\beta} = \\arg\\min_{\\beta} ||X\\beta - y||
  \\]

These trend features measure whether the variable has been increasing or decreasing over the last three years.
"""
))

cells.append(nbf.v4.new_code_cell(
"""def slope_3(values):
    x = np.array([0.0, 1.0, 2.0], dtype=float)
    y = np.asarray(values, dtype=float)
    return float(np.polyfit(x, y, 1)[0])

features = [
    "unemployment",
    "inflation",
    "gdp_growth",
    "life_expectancy",
    "population_growth",
]

ts_df = panel_df.copy()

for col in features:
    ts_df[f"{col}_change_1y"] = ts_df.groupby("country_code")[col].diff()
    ts_df[f"{col}_trend_3y"] = (
        ts_df.groupby("country_code")[col]
        .transform(lambda s: s.rolling(3).apply(lambda x: slope_3(np.array(x)), raw=False))
    )

ts_df["gdp_growth_next_year"] = ts_df.groupby("country_code")["gdp_growth"].shift(-1)
ts_df["downturn_next_year"] = (ts_df["gdp_growth_next_year"] < 0).astype(int)

ts_df.head(10)
"""
))

cells.append(nbf.v4.new_code_cell(
"""country_lookup = (
    ts_df[["country", "country_code"]]
    .drop_duplicates()
    .sort_values("country")
    .reset_index(drop=True)
)

country_lookup.head()
"""
))

cells.append(nbf.v4.new_markdown_cell(
"""## Select a country for diagnostics

Set the country name below. A few good starting options are:

- `"South Africa"`
- `"Brazil"`
- `"Kenya"`
- `"United States"`
- `"India"`
"""
))

cells.append(nbf.v4.new_code_cell(
"""selected_country = "South Africa"

country_row = country_lookup.loc[country_lookup["country"] == selected_country]
if country_row.empty:
    raise ValueError(
        f"Country '{selected_country}' not found. "
        "Choose a value from country_lookup['country']."
    )

selected_country_code = country_row["country_code"].iloc[0]
country_ts = ts_df.loc[ts_df["country_code"] == selected_country_code].copy()

print("Selected country:", selected_country)
print("Country code:", selected_country_code)
country_ts.head()
"""
))

cells.append(nbf.v4.new_markdown_cell(
"""## Helper functions for plotting

These helpers produce:

- level plots
- delta plots
- trend plots
- downturn overlays
"""
))

cells.append(nbf.v4.new_code_cell(
"""def downturn_marker_years(df):
    return df.loc[df["downturn_next_year"] == 1, "year"].tolist()


def add_downturn_markers(fig, df, y_col, name="Downturn next year"):
    downturn_years = downturn_marker_years(df)
    if downturn_years:
        marker_df = df.loc[df["year"].isin(downturn_years), ["year", y_col]].dropna()
        if not marker_df.empty:
            fig.add_trace(
                go.Scatter(
                    x=marker_df["year"],
                    y=marker_df[y_col],
                    mode="markers",
                    name=name,
                    marker=dict(size=10, symbol="diamond"),
                )
            )
    return fig


def plot_series(df, y_col, title, y_label):
    fig = px.line(
        df,
        x="year",
        y=y_col,
        markers=True,
        title=title,
    )
    fig.update_layout(
        xaxis_title="Year",
        yaxis_title=y_label,
        template="plotly_white",
        height=450,
    )
    fig = add_downturn_markers(fig, df, y_col)
    return fig


def plot_feature_triptych(df, feature, country_name):
    level_col = feature
    delta_col = f"{feature}_change_1y"
    trend_col = f"{feature}_trend_3y"

    fig_level = plot_series(
        df,
        level_col,
        f"{country_name}: {feature.replace('_', ' ').title()} level",
        feature.replace("_", " ").title(),
    )

    fig_delta = plot_series(
        df,
        delta_col,
        f"{country_name}: {feature.replace('_', ' ').title()} annual change",
        f"Δ {feature.replace('_', ' ').title()}",
    )

    fig_trend = plot_series(
        df,
        trend_col,
        f"{country_name}: {feature.replace('_', ' ').title()} 3-year trend",
        f"{feature.replace('_', ' ').title()} trend",
    )

    return fig_level, fig_delta, fig_trend
"""
))

cells.append(nbf.v4.new_markdown_cell(
"""## Plot levels, deltas, and 3-year trends

Run the next cell for any feature in:

- `unemployment`
- `inflation`
- `gdp_growth`
- `life_expectancy`
- `population_growth`
"""
))

cells.append(nbf.v4.new_code_cell(
"""feature_to_plot = "gdp_growth"

fig_level, fig_delta, fig_trend = plot_feature_triptych(
    country_ts,
    feature_to_plot,
    selected_country,
)

fig_level.show()
fig_delta.show()
fig_trend.show()
"""
))

cells.append(nbf.v4.new_markdown_cell(
"""## Quick multi-feature diagnostic loop

This cell plots all five indicators in sequence for the selected country.
"""
))

cells.append(nbf.v4.new_code_cell(
"""for feature in features:
    fig_level, fig_delta, fig_trend = plot_feature_triptych(
        country_ts,
        feature,
        selected_country,
    )
    fig_level.show()
    fig_delta.show()
    fig_trend.show()
"""
))

cells.append(nbf.v4.new_markdown_cell(
"""## Visual summary table for the selected country

This gives a compact numerical view of current level, recent annual change, and 3-year trend.
"""
))

cells.append(nbf.v4.new_code_cell(
"""latest_summary = country_ts.sort_values("year").tail(1).copy()

summary_rows = []
for feature in features:
    summary_rows.append(
        {
            "Feature": feature.replace("_", " ").title(),
            "Current value": latest_summary[feature].iloc[0],
            "Annual change": latest_summary[f"{feature}_change_1y"].iloc[0],
            "3-year trend": latest_summary[f"{feature}_trend_3y"].iloc[0],
        }
    )

summary_df = pd.DataFrame(summary_rows)
summary_df
"""
))

cells.append(nbf.v4.new_markdown_cell(
"""## Compare two countries on the same feature

This helps show whether downturn risk emerges from different levels, different short-term changes, or different medium-term trends.
"""
))

cells.append(nbf.v4.new_code_cell(
"""comparison_country = "Brazil"
comparison_row = country_lookup.loc[country_lookup["country"] == comparison_country]

if comparison_row.empty:
    raise ValueError(
        f"Country '{comparison_country}' not found. "
        "Choose a value from country_lookup['country']."
    )

comparison_code = comparison_row["country_code"].iloc[0]
comparison_ts = ts_df.loc[ts_df["country_code"] == comparison_code].copy()

comparison_feature = "gdp_growth"

plot_compare_df = pd.concat(
    [
        country_ts[["year", comparison_feature]].assign(country=selected_country),
        comparison_ts[["year", comparison_feature]].assign(country=comparison_country),
    ],
    ignore_index=True,
)

fig = px.line(
    plot_compare_df,
    x="year",
    y=comparison_feature,
    color="country",
    markers=True,
    title=f"{comparison_feature.replace('_', ' ').title()} comparison: {selected_country} vs {comparison_country}",
)
fig.update_layout(template="plotly_white", height=450)
fig.show()
"""
))

cells.append(nbf.v4.new_markdown_cell(
"""## Aggregate diagnostic across all countries

This gives a global view of whether downturn years differ systematically from non-downturn years.

The plot shows the average value of a chosen feature by year, split by whether the observation belongs to a downturn-next-year regime.
"""
))

cells.append(nbf.v4.new_code_cell(
"""aggregate_feature = "gdp_growth_change_1y"

aggregate_plot_df = (
    ts_df.groupby(["year", "downturn_next_year"], as_index=False)[aggregate_feature]
    .mean()
)

aggregate_plot_df["downturn_label"] = aggregate_plot_df["downturn_next_year"].map(
    {0: "No downturn next year", 1: "Downturn next year"}
)

fig = px.line(
    aggregate_plot_df,
    x="year",
    y=aggregate_feature,
    color="downturn_label",
    markers=True,
    title=f"Average {aggregate_feature.replace('_', ' ')} by year and downturn regime",
)
fig.update_layout(template="plotly_white", height=450)
fig.show()
"""
))

cells.append(nbf.v4.new_markdown_cell(
"""## Interpretation prompts

Use these prompts when writing up results:

### Levels
- Does the country enter downturn periods at unusually low GDP growth levels?
- Are unemployment and inflation levels structurally high before downturn years?

### Deltas
- Are downturns preceded by sudden negative changes in GDP growth?
- Do unemployment and inflation changes become unstable before downturns?

### Trends
- Are downturns associated with sustained negative 3-year GDP slopes?
- Does unemployment trend upward before downturn years?

### Comparison
- Do similar countries share the same level, delta, or trend patterns?
- Are trend features more informative than raw levels for early warning?
"""
))

cells.append(nbf.v4.new_markdown_cell(
"""## Suggested conclusion template

You can adapt the following:

> The time-series diagnostics suggest that downturn periods are better characterised by **deteriorating direction and sustained trends** than by levels alone. In particular, negative GDP-growth slopes and worsening annual changes appear before several downturn episodes, supporting the inclusion of delta and 3-year trend features in the predictive model.

This links the visual evidence back to your modelling choices.
"""
))

nb["cells"] = cells

out_path = Path("time_series_feature_diagnostics.ipynb")
with out_path.open("w", encoding="utf-8") as f:
    nbf.write(nb, f)

print(f"Created notebook at: {out_path}")
print(out_path.exists())


Created notebook at: time_series_feature_diagnostics.ipynb
True


## Step 1: Import Libraries & data loading

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

current = Path.cwd().resolve()
project_root = current.parent if current.name == "notebooks" else current

panel_final_path = project_root / "data" / "worldbank_panel_final.csv"
panel_fallback_path = project_root / "data" / "worldbank_panel.csv"

if panel_final_path.exists():
    panel_path = panel_final_path
elif panel_fallback_path.exists():
    panel_path = panel_fallback_path
else:
    raise FileNotFoundError(
        "Could not find input panel CSV. Expected one of:\n"
        f"- {panel_final_path}\n"
        f"- {panel_fallback_path}"
    )

panel_df = pd.read_csv(panel_path)
panel_df = panel_df.sort_values(["country_code", "year"]).reset_index(drop=True)

required_cols = [
    "country_code",
    "year",
    "unemployment",
    "inflation",
    "gdp_growth",
    "life_expectancy",
    "population_growth",
]

missing_cols = [c for c in required_cols if c not in panel_df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

if "country" not in panel_df.columns:
    panel_df["country"] = panel_df["country_code"]

panel_df.head()

,country,country_code,year,unemployment,inflation,gdp_growth,life_expectancy,population_growth
0,Afghanistan,AFG,2010,7.809,2.178538,14.362441,60.702,2.934687
1,Afghanistan,AFG,2011,7.830,11.804186,0.426355,61.250,3.691503
2,Afghanistan,AFG,2012,7.875,6.441213,12.752287,61.735,4.047863
3,Afghanistan,AFG,2013,7.921,7.385772,5.600745,62.188,3.418227
4,Afghanistan,AFG,2014,7.915,4.673996,2.724543,62.260,3.632519


## Step 2: Define Delta and Trend Features

In [4]:
def slope_3(values):
    x = np.array([0.0, 1.0, 2.0], dtype=float)
    y = np.asarray(values, dtype=float)
    return float(np.polyfit(x, y, 1)[0])

features = [
    "unemployment",
    "inflation",
    "gdp_growth",
    "life_expectancy",
    "population_growth",
]

ts_df = panel_df.copy()

for col in features:
    ts_df[f"{col}_change_1y"] = ts_df.groupby("country_code")[col].diff()
    ts_df[f"{col}_trend_3y"] = (
        ts_df.groupby("country_code")[col]
        .transform(lambda s: s.rolling(3).apply(lambda x: slope_3(np.array(x)), raw=False))
    )

ts_df["gdp_growth_next_year"] = ts_df.groupby("country_code")["gdp_growth"].shift(-1)
ts_df["downturn_next_year"] = (ts_df["gdp_growth_next_year"] < 0).astype(int)

ts_df.head(10)

,country,country_code,year,unemployment,inflation,gdp_growth,life_expectancy,population_growth,unemployment_change_1y,unemployment_trend_3y,inflation_change_1y,inflation_trend_3y,gdp_growth_change_1y,gdp_growth_trend_3y,life_expectancy_change_1y,life_expectancy_trend_3y,population_growth_change_1y,population_growth_trend_3y,gdp_growth_next_year,downturn_next_year
0,Afghanistan,AFG,2010,7.809,2.178538,14.362441,60.702,2.934687,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.426355,0
1,Afghanistan,AFG,2011,7.830,11.804186,0.426355,61.250,3.691503,0.021,NaN,9.625648,NaN,-13.936087,NaN,0.548,NaN,0.756816,NaN,12.752287,0
2,Afghanistan,AFG,2012,7.875,6.441213,12.752287,61.735,4.047863,0.045,0.0330,-5.362973,2.131338,12.325932,-0.805077,0.485,0.5165,0.356359,0.556588,5.600745,0
3,Afghanistan,AFG,2013,7.921,7.385772,5.600745,62.188,3.418227,0.046,0.0455,0.944559,-2.209207,-7.151542,2.587195,0.453,0.4690,-0.629636,-0.136638,2.724543,0
4,Afghanistan,AFG,2014,7.915,4.673996,2.724543,62.260,3.632519,-0.006,0.0200,-2.711776,-0.883608,-2.876201,-5.013872,0.072,0.2625,0.214292,-0.207672,1.451315,0
5,Afghanistan,AFG,2015,9.032,-0.661709,1.451315,62.270,3.119959,1.117,0.5555,-5.335705,-4.023740,-1.273229,-2.074715,0.010,0.0410,-0.512560,-0.149134,2.260314,0
6,Afghanistan,AFG,2016,10.116,4.383892,2.260314,62.646,2.535720,1.084,1.1005,5.045601,-0.145052,0.809000,-0.232115,0.376,0.1930,-0.584239,-0.548399,2.647003,0
7,Afghanistan,AFG,2017,11.184,4.975952,2.647003,62.406,2.808337,1.068,1.0760,0.592060,2.818830,0.386689,0.597844,-0.240,0.0680,0.272617,-0.155811,1.189228,0
8,Afghanistan,AFG,2018,11.192,0.626149,1.189228,62.443,2.910810,0.008,0.5380,-4.349802,-1.878871,-1.457775,-0.535543,0.037,-0.1015,0.102472,0.187545,3.911603,0
9,Afghanistan,AFG,2019,11.187,2.302373,3.911603,62.941,2.984389,-0.005,0.0015,1.676223,-1.336789,2.722375,0.632300,0.498,0.2675,0.073580,0.088026,-2.351101,1


## Step 3: Select Countries

In [5]:
country_lookup = (
    ts_df[["country", "country_code"]]
    .drop_duplicates()
    .sort_values("country")
    .reset_index(drop=True)
)

selected_country = "South Africa"

country_row = country_lookup.loc[country_lookup["country"] == selected_country]
if country_row.empty:
    raise ValueError(
        f"Country '{selected_country}' not found. "
        "Choose a value from country_lookup['country']."
    )

selected_country_code = country_row["country_code"].iloc[0]
country_ts = ts_df.loc[ts_df["country_code"] == selected_country_code].copy()

print("Selected country:", selected_country)
print("Country code:", selected_country_code)

Selected country: South Africa
Country code: ZAF


## Step 4: Define Plot helpers

In [6]:
def downturn_marker_years(df):
    return df.loc[df["downturn_next_year"] == 1, "year"].tolist()


def add_downturn_markers(fig, df, y_col, name="Downturn next year"):
    downturn_years = downturn_marker_years(df)
    if downturn_years:
        marker_df = df.loc[df["year"].isin(downturn_years), ["year", y_col]].dropna()
        if not marker_df.empty:
            fig.add_trace(
                go.Scatter(
                    x=marker_df["year"],
                    y=marker_df[y_col],
                    mode="markers",
                    name=name,
                    marker=dict(size=10, symbol="diamond"),
                )
            )
    return fig


def plot_series(df, y_col, title, y_label):
    fig = px.line(
        df,
        x="year",
        y=y_col,
        markers=True,
        title=title,
    )
    fig.update_layout(
        xaxis_title="Year",
        yaxis_title=y_label,
        template="plotly_white",
        height=450,
    )
    return add_downturn_markers(fig, df, y_col)


def plot_feature_triptych(df, feature, country_name):
    fig_level = plot_series(
        df,
        feature,
        f"{country_name}: {feature.replace('_', ' ').title()} level",
        feature.replace("_", " ").title(),
    )

    fig_delta = plot_series(
        df,
        f"{feature}_change_1y",
        f"{country_name}: {feature.replace('_', ' ').title()} annual change",
        f"Δ {feature.replace('_', ' ').title()}",
    )

    fig_trend = plot_series(
        df,
        f"{feature}_trend_3y",
        f"{country_name}: {feature.replace('_', ' ').title()} 3-year trend",
        f"{feature.replace('_', ' ').title()} trend",
    )

    return fig_level, fig_delta, fig_trend

## Step 5: Create Single Feature time-series Visualisations

In [7]:
feature_to_plot = "gdp_growth"

fig_level, fig_delta, fig_trend = plot_feature_triptych(
    country_ts,
    feature_to_plot,
    selected_country,
)

fig_level.show()
fig_delta.show()
fig_trend.show()

In [8]:
for feature in features:
    fig_level, fig_delta, fig_trend = plot_feature_triptych(
        country_ts,
        feature,
        selected_country,
    )
    fig_level.show()
    fig_delta.show()
    fig_trend.show()

In [9]:
comparison_country = "Brazil"
comparison_feature = "gdp_growth"

comparison_row = country_lookup.loc[country_lookup["country"] == comparison_country]
if comparison_row.empty:
    raise ValueError(
        f"Country '{comparison_country}' not found. "
        "Choose a value from country_lookup['country']."
    )

comparison_code = comparison_row["country_code"].iloc[0]
comparison_ts = ts_df.loc[ts_df["country_code"] == comparison_code].copy()

plot_compare_df = pd.concat(
    [
        country_ts[["year", comparison_feature]].assign(country=selected_country),
        comparison_ts[["year", comparison_feature]].assign(country=comparison_country),
    ],
    ignore_index=True,
)

fig = px.line(
    plot_compare_df,
    x="year",
    y=comparison_feature,
    color="country",
    markers=True,
    title=f"{comparison_feature.replace('_', ' ').title()} comparison: {selected_country} vs {comparison_country}",
)
fig.update_layout(template="plotly_white", height=450)
fig.show()

In [11]:
aggregate_feature = "gdp_growth_change_1y"

aggregate_plot_df = (
    ts_df.groupby(["year", "downturn_next_year"], as_index=False)[aggregate_feature]
    .mean()
)

aggregate_plot_df["downturn_label"] = aggregate_plot_df["downturn_next_year"].map(
    {0: "No downturn next year", 1: "Downturn next year"}
)

fig = px.line(
    aggregate_plot_df,
    x="year",
    y=aggregate_feature,
    color="downturn_label",
    markers=True,
    title=f"Average {aggregate_feature.replace('_', ' ')} by year and downturn regime",
)
fig.update_layout(template="plotly_white", height=450)
fig.show()

In [12]:
latest_summary = country_ts.sort_values("year").tail(1).copy()

summary_rows = []
for feature in features:
    summary_rows.append(
        {
            "Feature": feature.replace("_", " ").title(),
            "Current value": latest_summary[feature].iloc[0],
            "Annual change": latest_summary[f"{feature}_change_1y"].iloc[0],
            "3-year trend": latest_summary[f"{feature}_trend_3y"].iloc[0],
        }
    )

summary_df = pd.DataFrame(summary_rows)
summary_df

,Feature,Current value,Annual change,3-year trend
0,Unemployment,32.279000,0.181000,-0.494500
1,Inflation,4.361152,-1.714091,-1.339360
2,Gdp Growth,0.534844,-0.271261,-0.761647
3,Life Expectancy,66.312000,0.173000,0.429000
4,Population Growth,1.249514,-0.078587,-0.082229


In [13]:
def plot_multi_country_series(df, countries, feature, title=None):
    plot_df = df.loc[df["country"].isin(countries)].copy()

    fig = px.line(
        plot_df,
        x="year",
        y=feature,
        color="country",
        markers=True,
        title=title or f"{feature.replace('_', ' ').title()} by country",
    )

    fig.update_layout(
        template="plotly_white",
        xaxis_title="Year",
        yaxis_title=feature.replace("_", " ").title(),
        height=500,
    )

    return fig

In [ ]:
countries_to_plot = ["South Africa", "Brazil", "Kenya", "India"]

plot_multi_country_series(
    ts_df,
    countries=countries_to_plot,
    feature="gdp_growth",
    title="GDP growth comparison across countries"
).show()
